# [GIS] 집계구 단위 사회적 고립 위험 지수 v2 — 2026-10-20

**입력**: `data/processed/master_sgis2024_selected_no_industry_counts.csv` (SGIS 2024 집계구 통계)
**출력**: `data/interim/isolation_index_v2.csv`(집계구별 차원 비율 · z-score · 종합점수 · 지역 내 백분위 · 5등급), QGIS 자료형 파일 `.csvt`, 열 설명 `_dictionary.csv`

**v1 → v2 변경 이유** (2026-10-20 지도 검토): 강남 v1 5등급 155개 중 109개가 종사자 > 거주자인 업무·상업지였다.
원인은 ① 1인가구 = 고립 가정(오피스텔 청년 직장인) ② '주택이외 거처'에 오피스텔 포함 ③ 노후주택이 노후 아파트·상가를 구분 못함.

**v2 설계**
| 차원 | 정의 | 근거 |
| --- | --- | --- |
| SOLO50 중장년·고령 독거 | 1인가구율 × 50세+ 비율 | 1인가구×연령 교차표 대용. 복지부 고독사 위험군 50~60대 집중 |
| OLD 고령 | 65세+ 비율 — 노년부양비·노령화지수에서 역산(결측 없음) | AHR, Age UK, LSNS |
| MALE5064 중장년 남성 | 50~64세 남성 / 총인구 | 2024 고독사 실태조사(남성 81.7%, 50~60대 62.9%) |
| HOUSV 주거취약 | (비주거용 건물내 주택 + 20㎡ 이하) / (총주택 + 주택이외 거처) | 기본계획·2024 실태조사(고시원·원룸). 오피스텔 혼입 방지 위해 '주택이외 거처'는 분모에만 |
| OLDHO 노후 비아파트 | 1989년 이전 비율 × (단독+다세대+연립)/총주택 | 서울시 위험군 다가구 47%, 대구보건대 노후주거지 |

제외: 유동인구(→ H4 활동 신호층), 50㎡ 이상 연건평·건축년도 2000년대 이후·비친족가구(고립과 무관 또는 결측 과다).

**결측**: 모든 카운트 변수의 비결측 최솟값이 5 → 결측 = 0~4명 소량 비공개. 0으로 대치(민감도: 2).
**판단 보류**: 종사자/거주자 > 1(업무지) 또는 인구/가구 > 5(집단가구)는 등급 0. 원본 계산은 인구·가구·주택이 모두 양수인 집계구만 남기므로, 0 이하인 행은 결과에서 제외한다.
**점수화**: 차원별 z(±3 절단) 등가중 평균 → **지역(강남·춘천) 내** 백분위 5등급.

## 저장소 실행 안내

- **관련 이슈**: [#24 SGIS 지수 개선](https://github.com/Team-Hambugi-Lab/bigcontest-social-isolation/issues/24)
- **분석 목적**: 사용자 제공 v2 계산을 재현하고, #24의 v3 개선을 비교할 기준 코드를 공유한다.
- **원본 파일명**: `isolation_index_v2.ipynb`.
- **이번 변경**: 입력·출력 디렉터리, 코드 문자열 읽기, QGIS 보조파일 저장, 열 설명과 실행 안내. 차원·가중치·보류 임계값·민감도 계산은 원본대로 유지한다.
- **v3 상태**: 보류 3단계화·테마 A/B 분리·단지 유형 라벨은 아직 구현하지 않았다. 이 노트북을 v3 완료본으로 해석하지 않는다.
- **자료 기간**: SGIS 2024년 통계. 입력이 집계구×월이면 `STD_YM`의 첫 달만 사용한다.
- **주요 파라미터**: `IMPUTE=0`, `CLIP_Z=3`, `STRATIFY='REGION'`, `WORK_RATIO=1.0`, `GROUP_RATIO=5.0`; 5개 차원 등가중. 아래 설정 셀에서 확인한다.
- **난수 시드**: 난수를 쓰는 절차가 없어 해당 없음.
- **실행 환경**: Python 3.11 이상, 저장소의 pandas·numpy·scipy. `uv sync --extra dev` 후 저장소의 Python 환경으로 노트북을 실행한다.
- **작업 폴더**: 저장소 루트 또는 그 아래 `notebooks/`에서 실행한다. 루트를 찾아 `src.data.paths`의 `PROCESSED_DIR`, `INTERIM_DIR`을 사용한다.
- **입력 준비**: 팀 드라이브의 입력 마스터를 위 `data/processed/` 경로에 배치한다. 입력 CSV와 결과 데이터는 Git으로 추적하지 않는다.
- **해석 범위**: 지역 집계 대리지표이며 개인의 고립 여부나 실제 고령 독거 교차 인구를 직접 관측한 값이 아니다. 원본 결측·역산·제외 가정과 상단의 선행 수치·근거는 별도 분석 검토 대상이다.
- **실행 상태**: 원본에 저장된 실행 출력은 없다. 이번 공유 과정에서도 실데이터 결과를 재계산한 것으로 표시하지 않는다.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

pd.set_option("display.width", 180)


def find_project_root(start=None):
    """저장소 루트와 하위 작업 폴더에서 같은 데이터 경로를 찾는다."""
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "data" / "paths.py"
        ).is_file():
            return candidate
    raise FileNotFoundError("저장소 루트 또는 notebooks 폴더에서 노트북을 실행하세요.")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.paths import INTERIM_DIR, PROCESSED_DIR  # noqa: E402

# 입력 마스터는 팀 드라이브에서 받아 data/processed/에 배치한다.
SRC = PROCESSED_DIR / "master_sgis2024_selected_no_industry_counts.csv"
OUT = INTERIM_DIR / "isolation_index_v2.csv"
OUT_TYPES = OUT.with_suffix(".csvt")
OUT_DICTIONARY = OUT.with_name(f"{OUT.stem}_dictionary.csv")

# 원본 v2 분석 파라미터
IMPUTE = 0
CLIP_Z = 3
STRATIFY = "REGION"
WORK_RATIO = 1.0
GROUP_RATIO = 5.0
WEIGHTS = {"solo50": 1, "old": 1, "male5064": 1, "housv": 1, "oldho": 1}
S = "SGIS2024_"
DIMS = list(WEIGHTS)


## 1. 로드 (SGIS는 월별 동일 → 첫 달만)

In [ ]:
if not SRC.is_file():
    raise FileNotFoundError(
        f"입력 마스터를 {SRC.relative_to(ROOT)}에 배치한 뒤 실행하세요."
    )
df = pd.read_csv(
    SRC,
    dtype={"TOT_OA_CD": str, "ADM_CD": str, "STD_YM": str},
    low_memory=False,
)
one = df[df.STD_YM == df.STD_YM.min()].set_index('TOT_OA_CD')
sg = [c for c in one.columns if c.startswith(S)]
print('집계구', len(one), '| SGIS 변수', len(sg))
mins = one[sg].min()
print('카운트 변수 비결측 최솟값 분포:', mins[mins > 0].value_counts().head(3).to_dict(), '→ 결측 = 5 미만 소량 비공개')

## 2. 차원 계산

In [ ]:
def build(impute=IMPUTE, weights=None, strat=STRATIFY, clip=CLIP_Z):
    d = one.copy()
    d[sg] = d[sg].fillna(impute)
    pop, hh, ho = d[S+'to_in_001'], d[S+'to_ga_001'], d[S+'to_ho_001']
    valid = (pop > 0) & (hh > 0) & (ho > 0)
    d = d[valid].copy(); pop, hh, ho = pop[valid], hh[valid], ho[valid]
    age = lambda ids: d[[f'{S}in_age_{i:03d}' for i in ids]].sum(axis=1)

    # 65세+ : 노년부양비(65+/15~64), 노령화지수(65+/0~14)에서 역산 — 결측 없음
    odr, ai = d[S+'to_in_005'], d[S+'to_in_004']
    p65 = np.where((odr > 0) & (ai > 0), 1 / (1 + 100/odr.replace(0, np.nan) + 100/ai.replace(0, np.nan)), 0.0)
    p50 = age([11, 12, 13]) / pop + p65                      # 50세 이상
    dwell = ho + d[S+'ho_gb_006']                            # 총거처 = 주택 + 주택이외 거처
    nonapt = (d[S+'ho_gb_001'] + d[S+'ho_gb_002'] + d[S+'ho_gb_004']) / ho

    d['p_1in']    = d[S+'ga_sd_005'] / hh
    d['solo50']   = d['p_1in'] * p50
    d['old']      = p65
    d['male5064'] = age([41, 42, 43]) / pop
    d['housv']    = (d[S+'ho_gb_005'] + d[S+'ho_ar_001']) / dwell
    d['oldho']    = (d[S+'ho_yr_001'] + d[S+'ho_yr_002']) / ho * nonapt
    d['nonapt']   = nonapt
    d['work_ratio'] = d[S+'to_em_020'] / pop
    d['pop_per_hh'] = pop / hh

    d['flag_work']  = d.work_ratio > WORK_RATIO
    d['flag_group'] = d.pop_per_hh > GROUP_RATIO

    w = pd.Series(weights or WEIGHTS)
    grp = d.groupby(strat) if strat else None
    z = pd.DataFrame(index=d.index)
    for k in DIMS:
        if strat: z[k] = grp[k].transform(lambda s: (s - s.mean()) / s.std()).clip(-clip, clip)
        else:     z[k] = ((d[k] - d[k].mean()) / d[k].std()).clip(-clip, clip)
    d['score'] = (z * w).sum(axis=1) / w.sum()
    pct = d.groupby(strat)['score'].rank(pct=True) if strat else d['score'].rank(pct=True)
    d['pctile'] = pct * 100
    d['grade'] = pd.cut(d['pctile'], [0, 20, 40, 60, 80, 100], labels=[1, 2, 3, 4, 5]).astype(int)
    d.loc[d.flag_work | d.flag_group, 'grade'] = 0          # 판단 보류
    for k in DIMS: d['z_' + k] = z[k]
    return d

base = build()
print('포함', len(base), '| 보류(업무지)', base.flag_work.sum(), '| 보류(집단가구)', base.flag_group.sum())

## 3. 결과

In [ ]:
print('[차원 기술통계]'); display(base[DIMS].describe().T[['mean', '50%', 'max']].round(3))
print('[차원 간 Spearman]'); display(base[DIMS].corr('spearman').round(2))
print('[지역 × 등급, 0 = 보류]'); display(pd.crosstab(base.REGION, base.grade))
print('[지역 × 등급: 1인가구 수]'); display(pd.crosstab(base.REGION, base.grade, values=base[S+'ga_sd_005'], aggfunc='sum').astype(int))
g5 = base[base.grade == 5]
print('5등급 프로파일 (지역별 평균)'); display(g5.groupby('REGION')[['p_1in'] + DIMS].mean().round(3))

In [ ]:
print('[지역별 상위 10]')
for reg, sub in base[base.grade > 0].groupby('REGION'):
    display(sub.sort_values('score', ascending=False).head(10)[['ADM_CD', S+'to_in_001', S+'ga_sd_005', 'p_1in'] + DIMS + ['score']].round(3))

## 4. 차원 기여도 (feature importance 대용)

In [ ]:
z = base[['z_' + k for k in DIMS]].copy(); z.columns = DIMS
ok = base.grade > 0; g5 = base.grade == 5
imp = pd.DataFrame(index=DIMS)
imp['score 상관'] = [spearmanr(z.loc[ok, k], base.loc[ok, 'score']).correlation for k in DIMS]
for k in DIMS:
    s = z.drop(columns=k).mean(axis=1)
    p = s.groupby(base.REGION).rank(pct=True) if STRATIFY else s.rank(pct=True)
    imp.loc[k, '제거 시 5등급 유지율'] = ((p > 0.8) & g5).sum() / g5.sum()
imp['5등급 평균 기여'] = (z[g5] / len(DIMS)).mean()
zz = (z[ok] - z[ok].mean()) / z[ok].std(); _, sv, vt = np.linalg.svd(zz.values, full_matrices=False)
imp['PCA1 적재'] = vt[0] * np.sign(vt[0][0]); print('PCA1 설명분산', round(sv[0]**2 / (sv**2).sum(), 3))
imp.round(3)

## 5. 민감도

In [ ]:
def agree(a, b):
    j = a[['score', 'grade']].join(b[['score', 'grade']], rsuffix='_b'); j = j[(j.grade > 0) & (j.grade_b > 0)]
    return round(spearmanr(j.score, j.score_b).correlation, 3), f"{((j.grade == 5) & (j.grade_b == 5)).sum() / (j.grade == 5).sum():.0%}"
alts = {
    '결측 2 대치':      build(impute=2),
    '전체 통합 백분위':  build(strat=None),
    '노후 제외':        build(weights={**WEIGHTS, 'oldho': 0}),
    '주거취약 제외':    build(weights={**WEIGHTS, 'housv': 0}),
    '인구 3차원만':     build(weights={'solo50': 1, 'old': 1, 'male5064': 1, 'housv': 0, 'oldho': 0}),
    '독거 2배 가중':    build(weights={**WEIGHTS, 'solo50': 2}),
}
pd.DataFrame({k: agree(base, v) for k, v in alts.items()}, index=['Spearman', '5등급 일치']).T

## 6. 저장

In [ ]:
# 원본과 같은 결과 열·집계구 인덱스를 저장한다.
cols = ["REGION", "ADM_CD", "AREA_M2", S + "to_in_001", S + "to_ga_001",
        S + "ga_sd_005", "p_1in", "nonapt", "work_ratio", "pop_per_hh",
        "flag_work", "flag_group"] + DIMS + ["z_" + k for k in DIMS] + [
            "score", "pctile", "grade"
        ]
OUT.parent.mkdir(parents=True, exist_ok=True)
base[cols].to_csv(OUT, encoding="utf-8-sig", index_label="TOT_OA_CD")

# Path.replace()는 파일 이동이므로, 확장자는 with_suffix()로 변경한다.
types = ['"String"'] + [
    '"String"' if c in ("REGION", "ADM_CD", "flag_work", "flag_group")
    else '"Integer"' if c == "grade" else '"Real"'
    for c in cols
]
OUT_TYPES.write_text(",".join(types) + "\n", encoding="utf-8")

# 공통 공유 규칙에 맞춘 열 설명. 지역 대리지표와 개인 관측을 구분한다.
column_info = {
    "TOT_OA_CD": ("집계구 코드", "string", "SGIS 코드; 앞자리 0 보존"),
    "REGION": ("지역 이름", "string", "입력 지역 구분"),
    "ADM_CD": ("SGIS 행정동 코드", "string", "앞자리 0 보존; 행안부 코드와 구분"),
    "AREA_M2": ("입력 집계구 면적", "number", "m2"),
    S + "to_in_001": ("총인구", "number", "명"),
    S + "to_ga_001": ("일반가구 수", "number", "가구"),
    S + "ga_sd_005": ("1인가구 수", "number", "가구"),
    "p_1in": ("1인가구 수 / 일반가구 수", "number", "비율"),
    "nonapt": ("단독·연립·다세대 / 총주택", "number", "비율"),
    "work_ratio": ("종사자 수 / 총인구", "number", "배"),
    "pop_per_hh": ("총인구 / 일반가구 수", "number", "명/가구"),
    "flag_work": ("종사자비가 WORK_RATIO 초과", "boolean", "True/False"),
    "flag_group": ("인구/가구가 GROUP_RATIO 초과", "boolean", "True/False"),
    "solo50": ("1인가구율 × 추정 50세 이상 비율; 교차 가구수 직접 관측 아님", "number", "지역 대리지표"),
    "old": ("노년부양비·노령화지수로 역산한 65세 이상 비율", "number", "추정 비율"),
    "male5064": ("50~64세 남성 / 총인구", "number", "비율"),
    "housv": ("비주거용 건물내 주택과 20m2 이하 합 / 총거처", "number", "지역 비율; 분자 중복 가능"),
    "oldho": ("1989년 이전 비율 × 비아파트 비율; 실제 교차 주택수 직접 관측 아님", "number", "지역 대리지표"),
    "score": ("차원별 z-score의 가중 평균", "number", "표준화 점수"),
    "pctile": ("등급 0 처리 전 계산한 점수 백분위", "number", "0 초과 100 이하; 기본 지역 내"),
    "grade": ("지역 내 상대 등급; 두 보류 플래그 중 하나면 0", "integer", "0=보류; 1~5=상대 등급"),
}
for dim in DIMS:
    column_info[f"z_{dim}"] = (f"{dim} 표준화 후 절단", "number", f"-{CLIP_Z}~{CLIP_Z}")
dictionary_rows = []
for name in ["TOT_OA_CD", *cols]:
    description, data_type, unit = column_info[name]
    dictionary_rows.append({
        "COLUMN_NAME": name,
        "DESCRIPTION": description,
        "DATA_TYPE": data_type,
        "UNIT_OR_RANGE": unit,
        "MISSING_MEANING": (
            "원본 v2의 결측 대치·필터링 이후 값; 상수 차원의 표준화 등 계산 불가 시 결측. "
            "입력 행 제외는 등급 0과 별개이며 결과에서 빠짐"
        ),
    })
pd.DataFrame(dictionary_rows).to_csv(OUT_DICTIONARY, index=False, encoding="utf-8-sig")

print("saved:", OUT.relative_to(ROOT), "| CSV 행·열:", len(base), len(cols) + 1)
print("QGIS types:", OUT_TYPES.relative_to(ROOT))
print("dictionary:", OUT_DICTIONARY.relative_to(ROOT))
